In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy optuna


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, classification_report
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing (a3 original) ─────────────────────────────────────
def preprocess(df):
    df = df.copy()
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)
    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    df = df.fillna(-1)
    return df


X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 59), X_test: (8834, 59)


In [5]:
# ── Cell 5: ADVERSARIAL VALIDATION ───────────────────────────────────────────
# Goal: find out if train and test come from the same distribution
# Method:
#   Label train rows as 0, test rows as 1
#   Train a binary classifier to tell them apart
#   If AUC ≈ 0.5 → distributions match → OOF is reliable
#   If AUC >> 0.5 → distributions differ → OOF overstates real LB performance
#   If AUC is high, the top features causing it are the ones leaking

print('=' * 60)
print('ADVERSARIAL VALIDATION')
print('=' * 60)

# Combine train + test, label them
X_adv = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y_adv = np.array([0] * len(X_train) + [1] * len(X_test))  # 0=train, 1=test

adv_model = CatBoostClassifier(
    iterations    = 500,
    learning_rate = 0.05,
    depth         = 4,      # shallow — we just want signal, not overfit
    random_seed   = 42,
    verbose       = 0,
    eval_metric   = 'AUC',
    thread_count  = -1,
)

skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_aucs = cross_val_score(adv_model, X_adv, y_adv, cv=skf_adv,
                            scoring='roc_auc', n_jobs=-1)

print(f'\nAdversarial AUC scores per fold: {[round(a,4) for a in adv_aucs]}')
print(f'Mean AUC: {adv_aucs.mean():.4f} ± {adv_aucs.std():.4f}')
print()
if adv_aucs.mean() < 0.55:
    print('✓ AUC ≈ 0.5 — train and test are from the SAME distribution')
    print('  OOF is a reliable proxy for LB. Gap is due to noise, not distribution shift.')
elif adv_aucs.mean() < 0.65:
    print('⚠ AUC slightly above 0.5 — mild distribution difference')
    print('  Some features may differ between train/test. Check feature importance below.')
else:
    print('✗ AUC >> 0.5 — SIGNIFICANT distribution mismatch!')
    print('  Features causing this should be dropped or down-weighted.')

ADVERSARIAL VALIDATION

Adversarial AUC scores per fold: [np.float64(0.4826), np.float64(0.4816), np.float64(0.501), np.float64(0.5057), np.float64(0.5081)]
Mean AUC: 0.4958 ± 0.0114

✓ AUC ≈ 0.5 — train and test are from the SAME distribution
  OOF is a reliable proxy for LB. Gap is due to noise, not distribution shift.


In [6]:
# ── Cell 6: Adversarial feature importance ────────────────────────────────────
# Train the adversarial model on full data to get feature importances
# Features with HIGH importance in distinguishing train vs test
# are the ones causing distribution mismatch → candidates to drop

adv_model.fit(X_adv, y_adv)

fi = pd.Series(
    adv_model.get_feature_importance(),
    index=X_train.columns
).sort_values(ascending=False)

print('Top 20 features that DISTINGUISH train vs test:')
print('(High importance = this feature differs between train and test)')
print()
print(f'{"Feature":<35} {"Importance":>12}')
print('-' * 50)
for feat, imp in fi.head(20).items():
    flag = ' ← SUSPECT' if imp > 5 else ''
    print(f'{feat:<35} {imp:>12.3f}{flag}')

print(f'\nBottom 10 (most stable between train/test):')
for feat, imp in fi.tail(10).items():
    print(f'  {feat:<35} {imp:.3f}')

Top 20 features that DISTINGUISH train vs test:
(High importance = this feature differs between train and test)

Feature                               Importance
--------------------------------------------------
blood_cell_count                          11.609 ← SUSPECT
white_blood_cell_count                     7.106 ← SUSPECT
mother_age                                 5.488 ← SUSPECT
parent_age_gap                             4.846
father_age                                 4.838
defect_x_symptom                           4.277
age                                        4.248
symptom_defect_ratio                       3.432
abortion_cnt                               3.423
birth_asphyxia                             2.894
missing_count                              2.636
substance_abuse                            2.492
late_vs_early                              2.337
gender                                     2.143
father_defect                              2.026
place_birth           

In [7]:
# ── Cell 7: Drop suspect features + retrain ───────────────────────────────────
# Based on adversarial importance, drop features with importance > threshold
# These features look different in train vs test, so they hurt generalization

# Threshold: drop features with adversarial importance > 5
# (adjust based on what Cell 6 shows — if AUC was low, skip this cell)
ADV_THRESHOLD = 5.0

suspect_features = fi[fi > ADV_THRESHOLD].index.tolist()
print(f'Suspect features (adv importance > {ADV_THRESHOLD}):')
for f in suspect_features:
    print(f'  {f}: {fi[f]:.3f}')

if len(suspect_features) == 0:
    print('No suspect features found — distributions are clean!')
    print('Proceeding with all features.')
    X_train_clean = X_train.copy()
    X_test_clean  = X_test.copy()
else:
    print(f'\nDropping {len(suspect_features)} suspect features...')
    X_train_clean = X_train.drop(columns=suspect_features)
    X_test_clean  = X_test.drop(columns=suspect_features)
    print(f'Features remaining: {X_train_clean.shape[1]} (was {X_train.shape[1]})')

Suspect features (adv importance > 5.0):
  blood_cell_count: 11.609
  white_blood_cell_count: 7.106
  mother_age: 5.488

Dropping 3 suspect features...
Features remaining: 56 (was 59)


In [8]:
# ── Cell 8: Optuna hyperparameter search ──────────────────────────────────────
# Search for best CatBoost params that maximise OOF balanced accuracy
# Focus on: depth, l2_leaf_reg, random_strength, rsm
# These are the params we've never tuned — lr and iterations stay fixed

print('Running Optuna search (30 trials)...')
print('This will take ~15-20 minutes')

class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

# Use 5-fold for speed during search (full 10-fold for final model)
skf_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'iterations'           : 1000,
        'learning_rate'        : 0.03,
        'depth'                : trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg'          : trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_strength'      : trial.suggest_float('random_strength', 0.1, 3.0),
        'rsm'                  : trial.suggest_float('rsm', 0.5, 1.0),
        'bagging_temperature'  : trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'min_data_in_leaf'     : trial.suggest_int('min_data_in_leaf', 5, 30),
        'class_weights'        : class_weights,
        'early_stopping_rounds': 50,
        'eval_metric'          : 'Accuracy',
        'random_seed'          : 42,
        'verbose'              : 0,
        'thread_count'         : -1,
    }

    fold_scores = []
    for tr_idx, val_idx in skf_opt.split(X_train_clean, y_train):
        X_tr, X_val = X_train_clean.iloc[tr_idx], X_train_clean.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx],             y_train[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_pred = np.argmax(model.predict_proba(X_val), axis=1)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))

    return np.mean(fold_scores)


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest OOF BA (5-fold): {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

Running Optuna search (30 trials)...
This will take ~15-20 minutes


Best trial: 12. Best value: 0.398738: 100%|██████████| 30/30 [03:51<00:00,  7.73s/it]


Best OOF BA (5-fold): 0.3987
Best params:
  depth: 8
  l2_leaf_reg: 1.2554074561515052
  random_strength: 0.15300699009014024
  rsm: 0.6693037260566354
  bagging_temperature: 0.7561917100518147
  min_data_in_leaf: 23


In [9]:
# ── Cell 9: Final model with best params — 3 seeds x 10 folds ────────────────
best_params = study.best_params

print('Training final model with Optuna best params...')
print(f'Params: {best_params}')

all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test_clean), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test_clean), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_clean, y_train)):
        X_tr, X_val = X_train_clean.iloc[tr_idx], X_train_clean.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx],             y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            depth                 = best_params['depth'],
            l2_leaf_reg           = best_params['l2_leaf_reg'],
            random_strength       = best_params['random_strength'],
            rsm                   = best_params['rsm'],
            bagging_temperature   = best_params['bagging_temperature'],
            min_data_in_leaf      = best_params['min_data_in_leaf'],
            class_weights         = class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')
        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test_clean) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'FINAL OOF BA: {final_oof:.4f}')
print(f'Best run OOF: 0.3933  (LB 0.37127)')
print(f'Change:       {final_oof - 0.3933:+.4f}')
print(f"{'='*60}")

Training final model with Optuna best params...
Params: {'depth': 8, 'l2_leaf_reg': 1.2554074561515052, 'random_strength': 0.15300699009014024, 'rsm': 0.6693037260566354, 'bagging_temperature': 0.7561917100518147, 'min_data_in_leaf': 23}

======================================== SEED=42 ========================================
  Fold  1: BA=0.3718  best_iter=44
  Fold  2: BA=0.4215  best_iter=23
  Fold  3: BA=0.4189  best_iter=71
  Fold  4: BA=0.4042  best_iter=20
  Fold  5: BA=0.4142  best_iter=87
  Fold  6: BA=0.4338  best_iter=38
  Fold  7: BA=0.3609  best_iter=2
  Fold  8: BA=0.3756  best_iter=19
  Fold  9: BA=0.3952  best_iter=17
  Fold 10: BA=0.4357  best_iter=3
  OOF BA (seed=42): 0.4032 | mean=0.4032 ± 0.0251

======================================== SEED=7 ========================================
  Fold  1: BA=0.4138  best_iter=53
  Fold  2: BA=0.4124  best_iter=11
  Fold  3: BA=0.4125  best_iter=95
  Fold  4: BA=0.4191  best_iter=76
  Fold  5: BA=0.3673  best_iter=28
  Fold  

In [10]:
# ── Cell 10: Per-class recall ─────────────────────────────────────────────────
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
best_recall = {0:0.314, 1:0.390, 2:0.270, 3:0.348, 4:0.793,
               5:0.339, 6:0.498, 7:0.238, 8:0.604, 9:0.139}

oof_labels = np.argmax(all_oof_proba, axis=1)
report     = classification_report(y_train, oof_labels, output_dict=True)

print(f'OOF BA: {final_oof:.4f}\n')
print(f'{"Class":<5} {"Name":<16} {"Best run":>10} {"Now":>8} {"Δ":>7}')
print('-' * 52)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = best_recall[cls]
    delta = r - r_old
    flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>10.3f} {r:>8.3f} {delta:>+7.3f}{flag}')

OOF BA: 0.3930

Class Name               Best run      Now       Δ
----------------------------------------------------
0     레베르시                  0.314    0.401  +0.087 ← up
1     낭포성섬유증                0.390    0.364  -0.026
2     당뇨                    0.270    0.293  +0.023 ← up
3     리증후군                  0.348    0.317  -0.031
4     암                     0.793    0.759  -0.034
5     테이-삭스                 0.339    0.301  -0.038
6     혈색소침착증                0.498    0.480  -0.018
7     사립체근병종                0.238    0.253  +0.015 ← LOW
8     알츠하이머                 0.604    0.582  -0.022
9     확인안됨                  0.139    0.181  +0.042 ← up


In [11]:
# ── Cell 11: Save submission ──────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_adv_optuna.csv')

print('Saved: submission_adv_optuna.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nFinal OOF BA: {final_oof:.4f}')
print(f'Submit if OOF > 0.3933  (current best = 0.37127 LB)')
print(f'Expected LB ≈ OOF − 0.022')

Saved: submission_adv_optuna.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     503
1    1273
2     676
3    1389
4     267
5    1165
6    1023
7    1216
8     197
9    1125
Name: count, dtype: int64

Final OOF BA: 0.3930
Submit if OOF > 0.3933  (current best = 0.37127 LB)
Expected LB ≈ OOF − 0.022
